# 06 Problem 2a: robustness to degradations
T4 GPU, about 45 min. No retraining.

Real images online rarely stay untouched. Platforms re-compress them, resize them and sometimes smooth them. A detector that only works on pristine files is not much use.

**This notebook asks:** if we damage the test images a bit, do the 3 Problem 1 detectors still work?

**How:**
- Take the 8 test sets (500 real + 500 fake each).
- Make 14 versions of them: clean, JPEG at 5 quality levels, 3 resize factors, 2 blur levels, 2 noise levels, and a "social" combo (shrink to 75% + JPEG 75, roughly what a re-upload does).
- Every change hits real AND fake images the same way. Otherwise the model could just learn "damaged = fake".
- Score the saved models (`best.pt`, the CLIP and DINOv2 probes) on every version.

Refs: Wang et al. 2020 (arXiv 1912.11035), Grommelt et al. 2024 (arXiv 2403.17608), UnivFD (arXiv 2302.10174)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## 1. GPU check
CLIP ViT-L/14 has to look at 8000 images x 14 conditions. That needs a GPU.

In [ ]:
import torch

ALLOW_CPU = False   # CPU is way too slow, debug only
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    msg = ("NO GPU FOUND. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart "
           "and run all again.")
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image

from src import datasets as ds
from src import degrade as dg
from src import eda_utils as eu
from src import evaluate as ev
from src import models as mdl
from src import robustness as rb
from src.config import generator_names, get_logger

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("06_problem2_degradation", P.logs)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
GEN_ORDER = [g["folder"] for g in cfg["generators"]]
TRAIN_GEN = cfg["train_generator"]
THR = cfg["evaluation"]["threshold"]
FEAT = cfg["features"]
RCFG = cfg["training"]["resnet50"]
CONDS = cfg["problem2"]["conditions"]  # drop entries in config.yaml to save time
RES, FIG = P.results / "problem2", P.figures / "problem2"
PRED_DIR = RES / "preds"               # one csv per model x test set x condition, nb 08 reuses them
FEAT_ROOT = P.features / "problem2"
MODEL_ORDER = ["ResNet-50 (fine-tuned)", "CLIP ViT-L/14 + LR", "DINOv2 ViT-B/14 + LR"]
RESNET, CLIP, DINO = MODEL_ORDER


def save(fig, name):
    eu.savefig(fig, f"06_{name}", FIG)
    plt.show()


# sort by id so every run sees images in the same order (cached features depend on it)
test = ds.load_index(P.manifests).query("split_set == 'test'").sort_values("sample_id").reset_index(drop=True)
print(f"{len(test)} test images, {len(CONDS)} conditions: {list(CONDS)}")

### Copy test images to local disk
Reading 8000 small files from Drive over and over is slow. One copy to the Colab disk makes every later pass fast.

In [ ]:
# copy only files not already on the local disk
n = ds.stage_locally(test["processed_relpath"], P.project_root, P.work_dir)
ROOT = P.work_dir
print(f"Copied {n} new files")

## 2. What the degradations look like
One fake and one real image under every condition. We zoom into the center (96 px) so JPEG blocks, blur and noise are visible.

- **jpegXX:** re-save as JPEG at quality XX. Lower = more compression.
- **resizeX:** shrink (or enlarge) by X, then resize back to 224 px. Detail lost while small does not come back.
- **blurX:** Gaussian blur with strength (sigma) X.
- **noiseX:** random pixel noise, X = spread on the 0-255 scale.
- **social:** resize 0.75 then JPEG 75.

In [ ]:
# first fake and first real image of the SD1.4 test set
ex = pd.concat([test.loc[(test["generator"] == TRAIN_GEN) & (test["y"] == y)].iloc[[0]] for y in [1, 0]])
fig, axes = plt.subplots(2, len(CONDS), figsize=(1.35 * len(CONDS), 3.3))
for r, (_, row) in enumerate(ex.iterrows()):
    img = Image.open(ROOT / row["processed_relpath"]).convert("RGB")
    for c, (cond, steps) in enumerate(CONDS.items()):
        ax = axes[r, c]
        # apply the condition, then crop the middle 96 px so the damage is visible
        ax.imshow(dg.FixedDegrade(steps)(img, row["sample_id"]).crop((64, 64, 160, 160)))
        ax.axis("off")
        if r == 0:
            ax.set_title(cond, fontsize=8)
    axes[r, 0].text(-0.1, 0.5, "fake SD1.4" if row["y"] else "real", transform=axes[r, 0].transAxes,
                    rotation=90, ha="right", va="center", fontsize=9)
fig.tight_layout()
save(fig, "degradation_examples")

## 3. Score the 3 Problem 1 models on every condition
- **ResNet-50:** degraded image goes straight into the network.
- **CLIP / DINOv2:** compute the frozen features of the degraded image, then apply the saved logistic regression.

Each (model, condition) result is saved as its own csv in `results/problem2/preds/`. CLIP/DINOv2 features are saved in `features/problem2/`. If Colab disconnects, rerun and finished parts are skipped. Notebook 08 reuses these files too.

The "can only test a child process" messages are harmless (Python 3.13 + DataLoader).

In [ ]:
# ResNet-50 from notebook 03 (no retraining, only scoring)
resnet = mdl.build_resnet50(weights=None)
ck = torch.load(P.checkpoints / "resnet50" / "best.pt", map_location="cpu", weights_only=False)
resnet.load_state_dict(ck["model"])
resnet = resnet.to(DEVICE).eval()
# P(fake) for every test image under every condition, one csv per condition
all_preds = [rb.score_resnet(resnet, test, ROOT, CONDS, RESNET, "genimage", PRED_DIR, RCFG["crop_size"],
                             RCFG["batch_size"], RCFG["num_workers"], DEVICE, RCFG["amp"], log)]

In [ ]:
# on Python 3.13 the DataLoader may print "can only test a child process", harmless
feat_kw = dict(image_size=cfg["preprocess"]["image_size"], batch_size=FEAT["batch_size"],
               shard_size=FEAT["shard_size"], num_workers=FEAT["num_workers"], device=DEVICE)
# CLIP and DINOv2 are frozen: features of the degraded image -> saved logistic regression
for kind, name in [("clip", CLIP), ("dinov2", DINO)]:
    clf = joblib.load(P.checkpoints / f"linear_probe_{kind}.joblib")
    lazy = rb.LazyExtractor(kind, cfg)  # only loads the big model if some features are not cached yet
    all_preds.append(rb.score_probe(lazy, clf, test, ROOT, CONDS, name, "genimage", PRED_DIR, FEAT_ROOT, log, **feat_kw))
    lazy.free()  # free GPU memory before the next model
preds = pd.concat(all_preds, ignore_index=True)
preds.to_csv(RES / "degradation_predictions.csv", index=False)
print(preds.groupby(["model", "condition"], sort=False).size().unstack("model").head(3))

### Sanity check: clean condition = Problem 1 numbers
The "clean" version is just the normal test set. So it must give the same AUROCs as notebook 03. If it doesn't, something is wired wrong. Tiny gaps (< 0.001) come from GPU half-precision rounding.

In [ ]:
# AUROC, accuracy, F1, real/fake accuracy per model x condition x generator
res = rb.results_by_condition(preds, CONDS, GEN_ORDER, TRAIN_GEN, THR)
res.insert(3, "generator_name", res["generator"].map(NAMES).fillna(res["generator"]))
ev.save_table(res, RES / "degradation_results.csv")
# compare the clean condition with the notebook 03 results
p1 = pd.read_csv(P.results / "problem1_results.csv")
chk = res.loc[res["condition"] == "clean"].merge(p1[["model", "generator", "auroc"]], on=["model", "generator"],
                                                 suffixes=("", "_problem1"))
gap = (chk["auroc"] - chk["auroc_problem1"]).abs()
print(f"max |AUROC clean - AUROC problem 1| = {gap.max():.4f} over {len(chk)} rows (only fp16 noise expected)")

## 4. AUROC vs degradation level
One panel per type of damage. Left to right = more damage (for JPEG, quality goes down to the right). Dashed line = 0.5 = guessing.

- `mean_all` = average over all 8 generators.
- `SD1.4` = the generator the models were trained on.
- `mean_unseen` = average over the 7 generators not seen in training.

AUROC only checks if fakes get **higher scores than reals**. It does not care where the 0.5 cutoff is. Section 5 looks at the cutoff.

In [ ]:
fig = rb.plot_degradation_curves(res, "mean_all", "auroc", MODEL_ORDER, title="Mean AUROC over all 8 generators")
save(fig, "auroc_vs_degradation_mean_all")
fig = rb.plot_degradation_curves(res, TRAIN_GEN, "auroc", MODEL_ORDER, title=f"AUROC on {NAMES[TRAIN_GEN]} (seen in training)")
save(fig, "auroc_vs_degradation_sd14")
fig = rb.plot_degradation_curves(res, "mean_unseen", "auroc", MODEL_ORDER, title="Mean AUROC over the 7 unseen generators")
save(fig, "auroc_vs_degradation_mean_unseen")

In [ ]:
cond_order = list(CONDS)
for g in ["mean_all", TRAIN_GEN, "mean_unseen"]:
    piv = (res.loc[res["generator"] == g].pivot(index="model", columns="condition", values="auroc")
              .reindex(index=MODEL_ORDER, columns=cond_order))
    print(f"\nAUROC, {NAMES.get(g, g)}")
    display(piv.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0, axis=None))
    drop = piv.sub(piv["clean"], axis=0)  # negative = worse than clean
    print(f"Change vs clean, {NAMES.get(g, g)}")
    display(drop.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.4, vmax=0.4, axis=None))
    piv.to_csv(RES / f"degradation_auroc_{NAMES.get(g, g)}.csv")

### Per generator heatmaps
Every model x generator x condition in one picture. Green = good, red = at or below chance.

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(6 * len(MODEL_ORDER), 4), sharey=True)
for ax, m in zip(axes, MODEL_ORDER):
    piv = (res.loc[(res["model"] == m) & res["generator"].isin(GEN_ORDER)]
              .pivot(index="generator_name", columns="condition", values="auroc")
              .reindex(index=[NAMES[g] for g in GEN_ORDER], columns=cond_order))
    sns.heatmap(piv, ax=ax, vmin=0.3, vmax=1, cmap="RdYlGn", annot=True, fmt=".2f", annot_kws={"fontsize": 6}, cbar=False)
    ax.set(title=m, xlabel="", ylabel="")
    ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
save(fig, "auroc_heatmap_generator_x_condition")

## 5. Who gets the errors: reals or fakes?
Accuracy on SD1.4 at the fixed 0.5 cutoff, split by class:
- `fake_acc` = share of fakes called fake.
- `real_acc` = share of reals called real.

If `fake_acc` falls while `real_acc` stays high, damaged fakes are being called real. That is the dangerous direction for a detector.

In [ ]:
for metric in ["fake_acc", "real_acc"]:
    fig = rb.plot_degradation_curves(res, TRAIN_GEN, metric, MODEL_ORDER, title=f"{metric} on {NAMES[TRAIN_GEN]}")
    save(fig, f"{metric}_vs_degradation_sd14")

## Findings
Numbers from the run on all 8 test sets (8000 images).

**Sanity check**
- Clean condition matches notebook 03 to within 0.0001 AUROC. The pipeline is consistent.

**AUROC drops less than expected**
- On SD1.4 (seen), every model keeps AUROC above 0.89 under every condition. ResNet-50 goes from 0.991 to 0.950 at JPEG 50 and 0.961 on "social". Worst case is noise 10 (0.920) and blur 2 (0.930).
- So the models can still **rank** damaged fakes above damaged reals on the generator they know.

**But the 0.5 cutoff breaks**
- On SD1.4, ResNet-50 catches 96% of clean fakes. That falls to 48% at JPEG 50, 26% at blur 2 and 13% at noise 10. CLIP shows the same pattern (48%, 40%, 28%).
- Real accuracy stays at about 0.95-1.0 under every condition.
- So damage pushes every score towards "real". Damaged fakes slip through, while real photos are still called real. With a fixed cutoff, a simple re-save at JPEG 60 is enough to hide about half of the SD1.4 fakes from the ResNet.
- Practical meaning: a deployed detector needs its cutoff re-tuned on degraded data, or training that includes degradations (notebook 08).

**Unseen generators**
- CLIP stays the best model on unseen generators under every condition (mean unseen: 0.839 clean, 0.732 at JPEG 50, 0.730 on social).
- But CLIP loses the most. Its lead over ResNet-50 shrinks from +0.111 (clean) to about +0.04-0.05 (JPEG 50, social). Most of CLIP's drop already happens at JPEG 85 (0.839 to 0.768).
- ResNet-50 is hurt most by resizing, especially enlarging (resize 1.5: mean unseen 0.728 to 0.619; ADM 0.46, GLIDE 0.41, BigGAN 0.26). Enlarging smooths the image, which removes the fine detail the ResNet relies on.
- DINOv2 is the flattest (small drops), but it starts lowest. Noise is its weak spot (noise 10: mean unseen 0.707 to 0.621).
- Odd one: ResNet-50 on BigGAN goes from 0.42 (clean) up to 0.68 with noise 10. Noise wipes out the fine detail that made BigGAN look "extra real" to it, so it falls back towards guessing instead of being confidently wrong.

**What notebook 08 should fix**
- The cutoff collapse on compressed, blurred and noisy fakes (fake accuracy), more than AUROC.
- ResNet-50's weakness to resizing and blur.
- CLIP's drop under JPEG on unseen generators.

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()